# ⚙️ Notebook 2 — Premier classifieur : TF-IDF + Machine Learning classique

Dans les notebooks précédents, tu as découvert le phishing et exploré le jeu de données.
Maintenant, on passe aux choses sérieuses : **entraîner un modèle de machine learning**
capable de classer automatiquement un email comme sûr ou phishing.

Mais il y a un problème : un ordinateur ne comprend pas le texte directement.
Il ne comprend que les **nombres**. Il faut donc d'abord transformer le texte
en représentation numérique. C'est là qu'intervient le **TF-IDF**.

---

## Ce que tu vas apprendre dans ce notebook

1. Comment le **TF-IDF** transforme du texte en nombres.
2. Comment entraîner et comparer **3 modèles classiques** :
   - Régression Logistique
   - Naïve Bayes Multinomial
   - SVM Linéaire (Support Vector Machine)
3. Comment évaluer un modèle avec **accuracy, précision, rappel et F1-score**.
4. Comment lire une **matrice de confusion**.
5. Pourquoi la **précision** est particulièrement importante pour la détection de phishing.

---

## 0 — Mise en place

Chargeons les données d'entraînement, de validation et de test
que nous avons préparées.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

plt.rcParams.update({
    "figure.figsize": (10, 5),
    "font.size": 13,
    "axes.grid": True,
    "grid.alpha": 0.3,
})

train = pd.read_csv("data/train.csv")
val   = pd.read_csv("data/val.csv")
test  = pd.read_csv("data/test.csv")

print(f"Entra\u00eenement : {len(train):,} emails")
print(f"Validation   : {len(val):,} emails")
print(f"Test         : {len(test):,} emails")

---

## 1 — Qu'est-ce que le TF-IDF ?

**TF-IDF** signifie *Term Frequency – Inverse Document Frequency*
(fréquence du terme – fréquence inverse de document).

L'idée est simple :

- **TF (Term Frequency)** : combien de fois un mot apparaît dans un email donné.
  Plus un mot apparaît souvent dans un email, plus il est probablement important
  pour cet email.

- **IDF (Inverse Document Frequency)** : à quel point un mot est rare dans
  l'ensemble du jeu de données. Un mot qui apparaît dans tous les emails
  (comme \"the\") n'est pas très informatif. Un mot rare (comme \"suspended\")
  est beaucoup plus utile.

- **TF-IDF = TF × IDF** : le score final combine les deux. Un mot obtient
  un score élevé s'il est fréquent dans cet email **et** rare dans les autres.

### Exemple concret

| Mot | TF (dans un email) | IDF (dans tout le dataset) | TF-IDF |
|---|---|---|---|
| \"the\" | Élevé (très fréquent) | Très faible (présent partout) | **Faible** → peu utile |
| \"verify\" | Moyen | Élevé (assez rare) | **Élevé** → très utile |
| \"xylophone\" | Très faible | Très élevé (très rare) | **Faible** → trop rare pour être utile |

Le résultat : chaque email est transformé en un **vecteur de nombres**
(un nombre par mot du vocabulaire), et c'est ce vecteur que le modèle
de machine learning va utiliser pour apprendre.

### Appliquons le TF-IDF à nos données

La bibliothèque `scikit-learn` fait tout le travail pour nous.
On lui donne du texte brut, et elle renvoie les vecteurs numériques.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# On crée le vectoriseur TF-IDF
# max_features=10000 : on garde les 10 000 mots les plus fréquents
# stop_words='english' : on enlève automatiquement les mots vides anglais
vectorizer = TfidfVectorizer(max_features=10000, stop_words="english")

# On "entraîne" le vectoriseur sur les données d'entraînement
# puis on transforme chaque ensemble
X_train = vectorizer.fit_transform(train["text"])
X_val   = vectorizer.transform(val["text"])
X_test  = vectorizer.transform(test["text"])

y_train = train["label"]
y_val   = val["label"]
y_test  = test["label"]

print(f"Taille du vocabulaire : {len(vectorizer.get_feature_names_out()):,} mots")
print(f"Forme de X_train : {X_train.shape}")
print(f"  \u2192 {X_train.shape[0]} emails, chacun repr\u00e9sent\u00e9 par {X_train.shape[1]} nombres")

In [ ]:
# Regardons les mots avec les plus forts scores TF-IDF pour le premier email
feature_names = vectorizer.get_feature_names_out()
first_email_tfidf = X_train[0].toarray().flatten()

top_indices = first_email_tfidf.argsort()[-10:][::-1]
print("Les 10 mots avec le plus fort score TF-IDF dans le premier email :\n")
for idx in top_indices:
    if first_email_tfidf[idx] > 0:
        print(f"  {feature_names[idx]:20s}  score = {first_email_tfidf[idx]:.3f}")

---

## 2 — Les métriques d'évaluation

Avant d'entraîner nos modèles, comprenons comment on va mesurer leur
performance. On ne se contentera pas de l'**accuracy** (précision globale) :

| Métrique | Définition simple | Question à laquelle elle répond |
|---|---|---|
| **Accuracy** | % de prédictions correctes | \"Le modèle se trompe-t-il souvent ?\" |
| **Précision** | Parmi les emails signalés \"phishing\", combien le sont vraiment ? | \"Quand le modèle crie au loup, a-t-il raison ?\" |
| **Rappel** | Parmi les vrais phishing, combien ont été détectés ? | \"Le modèle rate-t-il beaucoup de phishing ?\" |
| **F1-score** | Moyenne harmonique de précision et rappel | \"Bon équilibre entre précision et rappel ?\" |

### Pourquoi la précision est cruciale ici

Rappelle-toi le Notebook 0 : un **faux positif** (email sûr signalé
comme phishing) signifie qu'un message légitime est bloqué.
Une haute **précision** signifie peu de faux positifs.

Un haut **rappel** signifie que le modèle attrape la plupart des
vrais phishing. L'idéal est d'avoir les deux élevés !

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

def evaluate_model(name, model, X, y_true):
    """\u00c9value un mod\u00e8le et affiche les m\u00e9triques."""
    y_pred = model.predict(X)
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec  = recall_score(y_true, y_pred)
    f1   = f1_score(y_true, y_pred)
    
    print(f"\n{'='*50}")
    print(f"  {name}")
    print(f"{'='*50}")
    print(f"  Accuracy  : {acc:.4f}  ({acc*100:.1f}%)")
    print(f"  Pr\u00e9cision : {prec:.4f}  ({prec*100:.1f}%)")
    print(f"  Rappel    : {rec:.4f}  ({rec*100:.1f}%)")
    print(f"  F1-score  : {f1:.4f}  ({f1*100:.1f}%)")
    
    return {"nom": name, "accuracy": acc, "precision": prec,
            "rappel": rec, "f1": f1, "y_pred": y_pred}

---

## 3 — Modèle n°1 : Régression Logistique

Malgré son nom, la **régression logistique** est un modèle de
**classification** (pas de régression !). C'est l'un des modèles les
plus simples et les plus utilisés.

**Comment ça marche (simplifié) :**
Le modèle attribue un **poids** à chaque mot. Les mots typiques du
phishing (comme \"verify\", \"urgent\") reçoivent des poids positifs.
Les mots typiques des emails sûrs reçoivent des poids négatifs.
Pour classer un nouvel email, il additionne les poids de tous ses
mots et décide : si la somme est positive → phishing, sinon → sûr.

In [ ]:
from sklearn.linear_model import LogisticRegression

model_lr = LogisticRegression(max_iter=1000, random_state=42)
model_lr.fit(X_train, y_train)

results_lr = evaluate_model("R\u00e9gression Logistique", model_lr, X_val, y_val)

### Quels mots sont les plus importants pour ce modèle ?

Regardons les mots avec les poids les plus forts (indicateurs de phishing)
et les plus faibles (indicateurs de sûr).

In [ ]:
coefficients = model_lr.coef_[0]
top_phishing_idx = coefficients.argsort()[-15:][::-1]
top_safe_idx = coefficients.argsort()[:15]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Mots phishing
words_ph = [feature_names[i] for i in reversed(top_phishing_idx)]
weights_ph = [coefficients[i] for i in reversed(top_phishing_idx)]
axes[0].barh(words_ph, weights_ph, color="#e74c3c", edgecolor="white")
axes[0].set_title("Mots les plus associ\u00e9s au phishing", fontsize=13)
axes[0].set_xlabel("Poids du mod\u00e8le")

# Mots sûrs
words_sf = [feature_names[i] for i in reversed(top_safe_idx)]
weights_sf = [abs(coefficients[i]) for i in reversed(top_safe_idx)]
axes[1].barh(words_sf, weights_sf, color="#2ecc71", edgecolor="white")
axes[1].set_title("Mots les plus associ\u00e9s au s\u00fbr", fontsize=13)
axes[1].set_xlabel("Poids du mod\u00e8le (valeur absolue)")

plt.tight_layout()
plt.show()

---

## 4 — Modèle n°2 : Naïve Bayes Multinomial

Le **Naïve Bayes** est un modèle basé sur les probabilités.

**Comment ça marche (simplifié) :**
Pour chaque mot, le modèle calcule la probabilité qu'il apparaisse dans
un email de phishing vs. un email sûr. Pour classer un nouvel email,
il multiplie les probabilités de tous les mots présents et choisit la
classe la plus probable.

On l'appelle \"naïve\" parce qu'il suppose que les mots sont **indépendants**
les uns des autres (ce qui n'est pas vrai en réalité, mais ça marche
quand même étonnamment bien !).

In [ ]:
from sklearn.naive_bayes import MultinomialNB

model_nb = MultinomialNB()
model_nb.fit(X_train, y_train)

results_nb = evaluate_model("Na\u00efve Bayes Multinomial", model_nb, X_val, y_val)

---

## 5 — Modèle n°3 : SVM Linéaire

Le **SVM** (Support Vector Machine) est un modèle qui cherche la meilleure
\"frontière\" pour séparer les deux classes.

**Comment ça marche (simplifié) :**
Imagine que chaque email est un point dans un espace à 10 000 dimensions
(une dimension par mot). Le SVM trace une \"ligne\" (en fait un hyperplan)
qui sépare au mieux les points \"sûr\" des points \"phishing\", en maximisant
la marge entre les deux groupes.

In [ ]:
from sklearn.svm import LinearSVC

model_svm = LinearSVC(max_iter=2000, random_state=42)
model_svm.fit(X_train, y_train)

results_svm = evaluate_model("SVM Lin\u00e9aire", model_svm, X_val, y_val)

---

## 6 — Comparaison des 3 modèles

Mettons les résultats côte à côte pour les comparer.

In [ ]:
all_results = [results_lr, results_nb, results_svm]

comparison = pd.DataFrame({
    "Mod\u00e8le": [r["nom"] for r in all_results],
    "Accuracy": [r["accuracy"] for r in all_results],
    "Pr\u00e9cision": [r["precision"] for r in all_results],
    "Rappel": [r["rappel"] for r in all_results],
    "F1-score": [r["f1"] for r in all_results],
})

# Affichage en pourcentages
display_df = comparison.copy()
for col in ["Accuracy", "Pr\u00e9cision", "Rappel", "F1-score"]:
    display_df[col] = display_df[col].apply(lambda x: f"{x*100:.1f}%")

print("\n\u250c" + "\u2500"*62 + "\u2510")
print("\u2502       COMPARAISON DES MOD\u00c8LES (Validation)              \u2502")
print("\u2514" + "\u2500"*62 + "\u2518\n")
print(display_df.to_string(index=False))

In [ ]:
metrics = ["Accuracy", "Pr\u00e9cision", "Rappel", "F1-score"]
model_names = [r["nom"] for r in all_results]
colors = ["#3498db", "#e67e22", "#9b59b6"]

x = np.arange(len(metrics))
width = 0.25

fig, ax = plt.subplots(figsize=(12, 5))

for i, (result, color) in enumerate(zip(all_results, colors)):
    values = [result["accuracy"], result["precision"],
              result["rappel"], result["f1"]]
    bars = ax.bar(x + i*width, [v*100 for v in values], width,
                  label=result["nom"], color=color, edgecolor="white")
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f"{val*100:.1f}", ha="center", fontsize=9, fontweight="bold")

ax.set_ylabel("Score (%)")
ax.set_title("Comparaison des 3 mod\u00e8les sur l'ensemble de validation")
ax.set_xticks(x + width)
ax.set_xticklabels(metrics)
ax.set_ylim(0, 105)
ax.legend()
plt.tight_layout()
plt.show()

---

## 7 — Matrice de confusion

La matrice de confusion montre **exactement où** le modèle se trompe.
Affichons-la pour chacun des 3 modèles.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, result, color in zip(axes, all_results, colors):
    cm = confusion_matrix(y_val, result["y_pred"])
    disp = ConfusionMatrixDisplay(cm, display_labels=["S\u00fbr", "Phishing"])
    disp.plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(result["nom"], fontsize=12)
    ax.set_xlabel("Pr\u00e9diction du mod\u00e8le")
    ax.set_ylabel("\u00c9tiquette r\u00e9elle")

plt.tight_layout()
plt.show()

### Comment lire la matrice de confusion

```
                    Prédiction du modèle
                    Sûr         Phishing
Étiquette   Sûr     VN           FP
réelle      Phishing FN           VP
```

- **VN (Vrai Négatif)** : email sûr correctement identifié → ✅
- **VP (Vrai Positif)** : phishing correctement détecté → ✅
- **FP (Faux Positif)** : email sûr signalé à tort comme phishing → ❌
- **FN (Faux Négatif)** : phishing qui a échappé au modèle → ❌

On veut que FP et FN soient les plus petits possible.

### Question pour toi

- Quel modèle a le moins de **faux positifs** ?
- Quel modèle a le moins de **faux négatifs** ?
- Si tu devais choisir un seul modèle pour un vrai filtre email, lequel choisirais-tu ? Pourquoi ?

---

## 8 — Évaluation finale sur l'ensemble de test

Jusqu'ici, on a évalué nos modèles sur l'ensemble de **validation**.
Maintenant, choisissons le meilleur et évaluons-le une dernière fois
sur l'ensemble de **test** (qu'il n'a jamais vu).

C'est l'équivalent de l'examen final !

In [ ]:
# Trouvons le meilleur modèle selon le F1-score
best_idx = max(range(len(all_results)), key=lambda i: all_results[i]["f1"])
best_name = all_results[best_idx]["nom"]
best_model = [model_lr, model_nb, model_svm][best_idx]

print(f"Meilleur mod\u00e8le (selon le F1 sur la validation) : {best_name}")
print(f"\n\u2500\u2500\u2500 \u00c9valuation finale sur l'ensemble de TEST \u2500\u2500\u2500")

results_test = evaluate_model(f"{best_name} (TEST)", best_model, X_test, y_test)

In [ ]:
# Matrice de confusion finale
fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_test, results_test["y_pred"])
disp = ConfusionMatrixDisplay(cm, display_labels=["S\u00fbr", "Phishing"])
disp.plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title(f"Matrice de confusion finale \u2014 {best_name}", fontsize=13)
ax.set_xlabel("Pr\u00e9diction du mod\u00e8le")
ax.set_ylabel("\u00c9tiquette r\u00e9elle")
plt.tight_layout()
plt.show()

print(f"\nRapport de classification d\u00e9taill\u00e9 :\n")
print(classification_report(y_test, results_test["y_pred"],
                            target_names=["S\u00fbr", "Phishing"]))

---

## 9 — Testons avec nos propres emails !

Amusons-nous un peu : écris tes propres emails (en anglais) et vois
ce que le modèle prédit.

In [ ]:
# ✏️ MODIFIE CES EMAILS ET AJOUTES-EN D'AUTRES SI TU VEUX !

my_emails = [
    "URGENT: Your account has been compromised! Click here immediately to verify your identity or your account will be suspended.",
    "Hi team, just a reminder about tomorrow's meeting at 2pm. Please bring your laptops.",
    "Congratulations! You've won a $500 Amazon gift card. Click the link below to claim your prize now.",
    "Dear customer, your package has been shipped and will arrive on Thursday. Track your order here.",
]

X_custom = vectorizer.transform(my_emails)
predictions = best_model.predict(X_custom)

print("Pr\u00e9dictions du mod\u00e8le :\n")
for email, pred in zip(my_emails, predictions):
    label = "\ud83d\udea8 PHISHING" if pred == 1 else "\u2705 S\u00dbR"
    preview = email[:80] + ("..." if len(email) > 80 else "")
    print(f"  {label:15s}  \u2502  {preview}")

---

## 10 — Réflexion et conclusion

Écris un court paragraphe répondant à ces questions :

1. **Quel modèle a le mieux fonctionné ?** Étais-tu surpris ?

2. **Quelles sont les limites du TF-IDF ?** Peut-il comprendre le sens
   d'une phrase, ou seulement compter des mots ?

3. **Si un email dit \"Your account is NOT suspended\",** le modèle TF-IDF
   comprendrait-il la négation ? Pourquoi ?

4. **Comment pourrait-on faire mieux ?** (Indice : on verra demain !)

*Écris ta conclusion ici :*

- ...
- ...
- ...
- ...

---

## Résumé

Dans ce notebook, tu as :

- Compris comment le **TF-IDF** transforme du texte en vecteurs numériques.
- Entraîné **3 modèles classiques** : Régression Logistique, Naïve Bayes, et SVM.
- Appris à évaluer avec **accuracy, précision, rappel et F1-score**.
- Lu des **matrices de confusion** pour comprendre les erreurs.
- Découvert les **mots les plus importants** pour le modèle.
- Testé le modèle avec tes **propres emails**.

Le TF-IDF + ML classique fonctionne déjà très bien ! Mais il a une
limitation : il traite chaque mot **indépendamment**, sans comprendre
le contexte ni l'ordre des mots.

**À suivre :** Dans le *Notebook 03*, nous utiliserons un **Transformer**
(DistilBERT) qui comprend le contexte des mots et peut potentiellement
faire encore mieux !